In [ ]:
# Loading dataset

In [7]:
%run data_loader_panel.py



Start of data summary


File loaded successfully.
rd_panel_data.csv loaded successfully as df_panel_data.

df_panel_data structure:

|   # | Column                                                |   Non‑Null Count | Dtype   |
|-----|-------------------------------------------------------|------------------|---------|
|   0 | Ticker                                                |          169,193 | str     |
|   1 | SimFinId                                              |          169,194 | int64   |
|   2 | Currency                                              |          169,194 | str     |
|   3 | Fiscal Year                                           |          169,194 | int64   |
|   4 | Fiscal Period                                         |          169,194 | str     |
|   5 | Report Date                                           |          169,194 | str     |
|   6 | Publish Date                                          |          169,194 | str     |
|   7 | Restated Date       

In [10]:
# Get unique companies and their reporting years
companies_by_year = df_panel_data.groupby('Fiscal Year')['Company Name'].unique()

# Find companies present in each year, dropping NaN values
companies_by_year_dict = {year: set(c for c in companies if pd.notna(c)) for year, companies in companies_by_year.items()}

# Sort years to compare consecutive years
sorted_years = sorted(companies_by_year_dict.keys())

# Find companies that stopped reporting
disappeared_companies = {}
for i in range(len(sorted_years) - 1):
    current_year = sorted_years[i]
    next_year = sorted_years[i + 1]
    
    # Companies in current year but not in next year
    stopped = companies_by_year_dict[current_year] - companies_by_year_dict[next_year]
    
    if stopped:
        disappeared_companies[f"{current_year} -> {next_year}"] = sorted(list(stopped))

# Display companies that disappeared
for period, companies in disappeared_companies.items():
    print(f"\nCompanies that stopped reporting ({period}): {len(companies)}")
    print(companies[:10])  # Show first 10

# Create a summary dataframe of disappearing companies with their last year data
disappeared_summary = []
for company in {c for companies in disappeared_companies.values() for c in companies}:
    company_data = df_panel_data[df_panel_data['Company Name'] == company]
    last_year = company_data['Fiscal Year'].max()
    last_record = company_data[company_data['Fiscal Year'] == last_year].iloc[0] if len(company_data) > 0 else None
    if last_record is not None:
        disappeared_summary.append({
            'Company Name': company,
            'Last Report Year': int(last_year),
            'Revenue': last_record['Revenue'],
            'Operating Income': last_record['Operating Income (Loss)'],
            'Pretax Income': last_record['Pretax Income (Loss), Adj.'],
            'Net Cash from Operating': last_record['Net Cash from Operating Activities']
        })

disappeared_df = pd.DataFrame(disappeared_summary)
print("\n\nCompanies that stopped reporting - Last year metrics:")
print(disappeared_df.sort_values('Last Report Year', ascending=False).head(20))


Companies that stopped reporting (2008 -> 2009): 1
['Pennsylvania Real Estate Investment Trust']

Companies that stopped reporting (2010 -> 2011): 4
['COCA-COLA REFRESHMENTS USA, INC.', 'General Growth Properties Inc.', 'MF Global Holdings Ltd.', 'McAfee, Inc.']

Companies that stopped reporting (2011 -> 2012): 2
['CAMECO CORP', 'HEMACARE CORP /CA/']

Companies that stopped reporting (2012 -> 2013): 18
['AGNICO EAGLE MINES LTD', 'BARCLAY ROAD, INC.', 'BLUE COAT SYSTEMS INC', 'CYBRA CORP', 'Digital Domain Media Group, Inc.', 'Infusion Brands International, Inc.', 'JDA SOFTWARE GROUP INC', 'MICRONETICS INC', 'PEOPLES EDUCATIONAL HOLDINGS', 'RDA Holding Co.']

Companies that stopped reporting (2013 -> 2014): 22
['Actavis, Inc.', 'Aramark Services, Inc.', 'BLUE RIDGE REAL ESTATE CO', 'CelLynx Group, Inc.', 'DGT Holdings Corp.', 'EMERITUS CORP\\WA\\', 'EMTEC INC/NJ', 'Endo International', 'Forest Laboratories, LLC', 'GARDNER DENVER INC']

Companies that stopped reporting (2014 -> 2015): 19

In [15]:
# Inspect fiscal period structure to see if the data is quarterly
print("Fiscal Period value counts:")
print(df_panel_data['Fiscal Period'].value_counts(dropna=False))

print("\nUnique Fiscal Period values:")
print(sorted(df_panel_data['Fiscal Period'].dropna().unique()))
year_values = pd.to_numeric(df_panel_data['Fiscal Year'], errors='coerce').dropna().astype(int)
if len(year_values):
    print(f"\nData covers Fiscal Year {year_values.min()} through {year_values.max()}")
    if 2025 in year_values.unique():
        fy2025 = df_panel_data[year_values == 2025].copy()
        print(f"\nFiscal Year 2025 observations: {len(fy2025)}")

        for col in ['Report Date', 'Date']:
            if col in fy2025.columns:
                fy2025[col] = pd.to_datetime(fy2025[col], errors='coerce')

        sort_cols = []
        sort_asc = []
        if 'FiscalPeriodNum' in fy2025.columns:
            sort_cols.append('FiscalPeriodNum')
            sort_asc.append(True)
        if 'Report Date' in fy2025.columns:
            sort_cols.append('Report Date')
            sort_asc.append(True)
        if 'Date' in fy2025.columns:
            sort_cols.append('Date')
            sort_asc.append(True)

        last_2025 = fy2025.sort_values(
            sort_cols or ['Fiscal Period'],
            ascending=sort_asc or [True],
            na_position='last'
        ).iloc[-1]

        print("\nLast observation in Fiscal Year 2025:")
        print(last_2025[['Ticker', 'Company Name', 'Fiscal Year', 'Fiscal Period', 'FiscalPeriodNum', 'Report Date', 'Date']])
        print(
            f"Ends at Fiscal Period {last_2025.get('Fiscal Period')} "
            f"(FiscalPeriodNum {last_2025.get('FiscalPeriodNum')}), "
            f"Report Date {last_2025.get('Report Date')}, "
            f"Date {last_2025.get('Date')}."
        )
else:
    print("\nNo valid Fiscal Year values found.")
if 'FiscalPeriodNum' in df_panel_data.columns:
    print("\nUnique FiscalPeriodNum values:")
    print(sorted(df_panel_data['FiscalPeriodNum'].dropna().unique()))

Fiscal Period value counts:
Fiscal Period
Q4    43227
Q1    42607
Q3    41819
Q2    41541
Name: count, dtype: int64

Unique Fiscal Period values:
['Q1', 'Q2', 'Q3', 'Q4']

Data covers Fiscal Year 1999 through 2025

Fiscal Year 2025 observations: 2663

Last observation in Fiscal Year 2025:
Ticker                              ZS
Company Name             Zscaler, Inc.
Fiscal Year                       2025
Fiscal Period                       Q3
FiscalPeriodNum                      3
Report Date        2025-04-30 00:00:00
Date                               NaT
Name: 168892, dtype: object
Ends at Fiscal Period Q3 (FiscalPeriodNum 3), Report Date 2025-04-30 00:00:00, Date NaT.

Unique FiscalPeriodNum values:
[np.int64(1), np.int64(2), np.int64(3), np.int64(4)]
